## Logistic Regression via Gradient Descent

Suppose we have training data $\{(x_{11},\ldots, x_{1p}, y_1), \ldots, (x_{n1},\ldots, x_{np}, y_n)\}$.  So $(x_{i1}, \ldots x_{ip})$ is a feature observation and $y_i$ is the associated label.  We will assume that the $x_{ij}$ are real numbers and that $y_i \in \{0,1\}$ - so this is a binary classification problem.  The logistic regression model is defined as:

$$
\begin{align}
\hat z_i & = \beta_0 + \sum_{k=1}^{p} x_{ik}\,\beta_k \\
\hat y_i &= \sigma(z_i) = \frac{1}{1 - e^{-\hat{z}_i}}
\end{align}
$$

We will refer to $\sigma$ as the *sigmoid activation function*.

It is both mathematically useful and useful for implementing efficient vectorized code with packages like **numpy** and **PyTorch** to express the above set of $n$ equations as a matrix computation. Let $\beta = (\beta_0, \beta_1, \dots, \beta_p)^{T}$. Futhermore, arrange the training data into a matrices $X$, $Y$, $Z$ as follows (note the column of $1$s at the beginning of $X$ to absorb the intercept $\beta_{0}$)

$$
X = \begin{bmatrix} 1 & x_{11} & \cdots & x_{1p} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{np} \end{bmatrix}
$$

$$
Y = \begin{bmatrix} \, y_1 \, \\ \vdots \\ \, y_n \, \end{bmatrix}
$$


Now we can compute the a $n \times 1$ matrix of all the predictions $\hat{y_i}$ as with the following matrix computation:

$$
\sigma \big(X \beta \, \big) = \begin{bmatrix} \, \sigma(\hat z_1) \, \\ \vdots \\ \, \sigma(\hat z_n) \, \end{bmatrix}
$$

To borrow some language from deep learning, we will sometimes refer to $\sigma \big(X \beta \, \big)$ as a *full-batch forward pass* of the model.  We will sometimes use the following notation: $\hat{Y} = \sigma \big(X \beta \, \big)$ and $\hat{Z} = X \beta$.

## Generating Synthetic Data

In [1]:
import numpy as np

In [2]:
rng = np.random.default_rng(0)
n = 500
beta = np.array([1, -1, -0.5, 1.5])
intercept = 0

X_inputs = rng.normal(size=(n, beta.shape[0]))

z = intercept + X_inputs @ beta          # linear predictor (logits), shape (n,)
p = 1 / (1 + np.exp(-z))                 # sigmoid -> P(y_i = 1 | x_i)
y = rng.binomial(1, p)  

## Implementing Data Matrix and Parameters in Code

Below we have that $\beta_0=$ `bias` and $(\beta_1, \dots, \beta_p)=$ `weights`.  Again we are using terms that foreshadow neural netorks.

In [3]:
bias = rng.normal(scale=1, size=1)
weights = rng.normal(scale=1, size=4)
print(weights)
print(bias)

[-1.19235168 -0.01987898 -1.59969624 -0.59192106]
[0.11680682]


It will be useful concatenat the `weight` and `bias` into a single variable `parameters`$=(\beta_0, \beta_1, \dots, \beta_p) = \beta^T$.

In [4]:
parameters = np.append(bias, weights)
parameters

array([ 0.11680682, -1.19235168, -0.01987898, -1.59969624, -0.59192106])

Let's now construct our data matrix `X`$=X$.

In [5]:
ones = np.ones((X_inputs.shape[0], 1), dtype=X_inputs.dtype)
X = np.hstack((ones, X_inputs))
X

array([[ 1.        ,  0.12573022, -0.13210486,  0.64042265,  0.10490012],
       [ 1.        , -0.53566937,  0.36159505,  1.30400005,  0.94708096],
       [ 1.        , -0.70373524, -1.26542147, -0.62327446,  0.04132598],
       ...,
       [ 1.        ,  1.85744412, -0.82073139, -0.14101172,  1.35408138],
       [ 1.        ,  1.73963397,  0.06523589,  1.24972527,  0.75063397],
       [ 1.        , -0.55581573, -2.01881162, -0.90942756,  0.36922933]],
      shape=(500, 5))

## Defining Likelihood and Binary Cross-Entropy

We'll want a measure of how well the logistic regression model is working for a given set of parameter values. Towards this end we define some additional notation:

1. The *log-likelihood* of the $i$th prediction will be $l_i$.
1. The overall *loss* for a give set of paramenters will be $L(\beta_0, \beta_1, \dots, \beta_p)$ - this will be binary cross-entropy.

The definitions are as follows

$$
l_i = \, y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)
$$

$$
L(\beta_0, \beta_1, \dots, \beta_p)= -\frac{1}{n}\sum_{i=1}^n l_i
$$

## Implementing Binary Cross-Entropy In Code

The above definitions of log-likelihood and binary cross-entropy are implemented in the following `loss()` function:

In [6]:
def sigmoid(t):
    """When applied to an array, applies elementwise"""
    return 1 / (1 + np.exp(-t))

In [7]:
def loss(parameters, X, y):
    """Binary cross-entropy: -(1/n)[y^T log(y_hat) + (1-y)^T log(1-y_hat)]"""
    z_hat = X @ parameters
    y_hat = sigmoid(z_hat)                  
    return -(y @ np.log(y_hat) + (1 - y) @ np.log(1 - y_hat)) / len(y)

Regarding the `sigmoid()` and `loss()` functions:

1. In **numpy**, `np.exp()` when applied to an `array` is applied to each element.
2. `X @ parameters` is a matrix computation that results in a $n \times 1$ matrix consisting of all the logits.
3. Due to #1, when `sigmod()` is applied to an `array` it is applied to each element.
4. `y @ np.log(y_hat)` and `(1 - y) @ np.log(1 - y_hat)`, though they use the matrix multiplication operator `@` are just dot products which faciltate the efficient calculation all the log-likelihood and hence binary cross-entropy. 

In [8]:
loss(parameters, X, y)

np.float64(1.2954827887413671)

The following implementation of binary cross-entropy is more numerically stable.  **UNDERSTAND WHY**

In [9]:
def loss_stable(parameters, X, y):
    """Same loss, rewritten as mean(log(1 + e^z) - y*z) with z = Xw."""
    z_hat = X @ parameters
    return np.mean(np.logaddexp(0, z_hat) - y * z_hat)

In [10]:
loss_stable(parameters, X, y)

np.float64(1.2954827887413671)

## Deriving the Gradient of the Loss

$$
\nabla {L}(\beta_0, \ldots, \beta_p) = \frac{1}{n} X^{T} \big(\hat{Y} - Y \big)
$$

In [11]:
def gradient(w, X, y):
    """Gradient of BCE: (1/n) X^T (sigmoid(Xw) - y)"""
    z_hat = X @ parameters
    y_hat = sigmoid(z_hat)
    return X.T @ (y_hat - y) / len(y)

## Deriving The Parameter Update

## Implementing the Parameter Update - Fitting the Model

The parameter update procedure is implemented in the following `for` loop for 100 iterations (`epochs`).

In [12]:
learning_rate = 0.1
for epoch in range(1000):
    grad = gradient(parameters, X, y)
    parameters += -learning_rate * grad

In [13]:
parameters

array([ 0.00727136,  1.03551377, -0.99811375, -0.55674278,  1.44715073])

In [14]:
[intercept] + list(beta)

[0, np.float64(1.0), np.float64(-1.0), np.float64(-0.5), np.float64(1.5)]

## Comparing Gradient Descent to `sklearn`

Our results match `sklearn` very closely.

In [15]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression().fit(X_inputs, y)
print(model.intercept_, model.coef_)

[0.01093503] [[ 1.00300032 -0.96417731 -0.53883536  1.40013103]]


## Implementing in **PyTorch**